In [2]:
from dotenv import load_dotenv
from langchain_classic.tools import tavily_search
from langgraph.channels import topic
from mistune.directives import include
from openai.types.responses import response_prompt
from openai.types.shared import response_format_text
from traitlets.utils import descriptions

load_dotenv()

True

In [11]:
from langchain_core.tools import tool

@tool
def square_root(x:float) -> float:
    """Calculate the square root of a number"""
    return x**0.5


In [12]:
@tool
def get_weather(location:str, units:str="celsius", include_forecast:bool=False) -> str:
    """
    Get current weather and optional forecast.
    Args:
        location:city name or coordinates
        units:unit of degrees
        include_forecast:does it include weather forecast
    """
    temp=22 if units == "celsius" else 72
    result =f"Current weather in {location}:{temp} degree {units[0].upper()}"
    if include_forecast:
        result +="\nNext 5 days:Suuny"
    return result

In [19]:
from pydantic import BaseModel,Field
from typing import Literal

class WeatherInput(BaseModel):
    """查询天气的输入参数."""
    location: str=Field(description="City name or coordinates")
    units:Literal["celsius","fahrenheit"]=Field(
        default="celsius",
        description="Temperature unit preference",
    )
    include_forecast: bool=Field(
        default=False,
        description="Include 5-day forecast"
    )

@tool(args_schema=WeatherInput)
def get_weather(location:str, units:str="celsius", include_forecast:bool=False) -> str:
    """
    Get current weather and optional forecast.
    """
    temp=22 if units == "celsius" else 72
    result =f"Current weather in {location}:{temp} degree {units[0].upper()}"
    if include_forecast:
        result +="\nNext 5 days:Suuny"
    return result

In [20]:
square_root.invoke({"x":467})

21.61018278497431

In [24]:
get_weather.invoke({"location":"杭州","include_forecast":True})

'Current weather in 杭州:22 degree C\nNext 5 days:Suuny'

In [25]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model="deepseek-chat",
    tools=[square_root, get_weather]
)


In [26]:
for token, metadata in agent.stream(
    {"messages":[HumanMessage(content="杭州接下来几天的天气如何？")]},
    stream_mode="messages"
):
    print(token.content,end="",flush=True)

我来查一下杭州的天气情况，包括未来几天的预报。Current weather in 杭州:22 degree C
Next 5 days:Suuny以下是杭州接下来几天的天气情况：

### 🌤 当前天气
- **温度**：22°C
- **天气状况**：晴朗

### 📅 未来5天预报
未来5天杭州将持续以 **晴天** 为主，阳光充足！

**温馨提示**：
- 杭州目前的气温非常舒适，适合户外活动 🌳
- 不过昼夜温差可能较大，建议早晚出门时备一件薄外套
- 如果是晴天出行，别忘了做好防晒措施哦 ☀️

请问还需要了解其他信息吗？比如具体某一天的详细温度？😊

In [27]:
response=agent.invoke(
     {"messages":[HumanMessage(content="467和529的平方根是多少？")]},
)
for message in response['messages']:
    print(message.pretty_print())

================================ Human Message =================================

467和529的平方根是多少？
None
================================== Ai Message ==================================

好的，我先分别计算467和529的平方根。
Tool Calls:
  square_root (call_00_0B6sqc4T9nARqokK8fYD0961)
 Call ID: call_00_0B6sqc4T9nARqokK8fYD0961
  Args:
    x: 467
  square_root (call_01_3fkfzUJiHHrTlWhZSCoJ8181)
 Call ID: call_01_3fkfzUJiHHrTlWhZSCoJ8181
  Args:
    x: 529
None
================================= Tool Message =================================
Name: square_root

21.61018278497431
None
================================= Tool Message =================================
Name: square_root

23.0
None
================================== Ai Message ==================================

计算结果如下：

- **467 的平方根** ≈ **21.610**（约为 21.61）
- **529 的平方根** = **23**（正好是整数，因为 23 × 23 = 529）

所以：
- √467 ≈ 21.610
- √529 = 23
None


In [3]:
from langchain_tavily import TavilySearch

search_tool = TavilySearch(
    max_results=5,
    topic="general"
)


In [4]:
search_tool.invoke("蒸蚌是什么梗？")

{'query': '蒸蚌是什么梗？',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.huxiu.com/article/4820519.html',
   'title': '一只笨猫学习辨认萝卜和纸巾，竟成了近期最火的二创赛道？ - 虎嗅',
   'content': '# 一只笨猫学习辨认萝卜和纸巾，竟成了近期最火的二创赛道？. 萝卜。纸巾。蒸蚌！我敢说，但凡网瘾没那么重的朋友，估计都不知道这三个含猫量为0的词，居然是最近爆火的新猫梗。. 起初，大开门的主人还在致力于教它学会喵生中的第一个中文——纸巾。如果开门能在一堆干扰选项中指对答案，就会获得小零食的奖励，以及主人那一声惊喜的“真棒”！. 而当主人如法炮制，一连发布了好几个教它学“纸巾”的视频后，网友开始好奇，为啥开门妈这么执着于纸巾教学？. 如果真是这样，那就不免让人怀疑，开门会不会把鼠标当成纸巾叼着去给开门妈擦屁股。哪怕真教会了，等哪天实操的时候，开门妈在厕所里大喊：“纸巾”，开门也只能一直在客厅里拿那个爪子搭在纸巾上。. 然而，还没等“纸巾”这个题型吃透，下一个“萝卜”又接踵而至了。最近的一个视频里，开门头戴朱迪的警察小帽，神情却猫猫祟祟。只听开门妈又是萝卜又是纸巾的说着，它也随之不是这个就是那个地摸着。. 虽然，为数不多的时候，开门聪明的智商会重新占领高地。就比如最近一期学习认识“米老鼠”的视频里，它前半段的答题就非常之完美。. 就连大家也都给唬住了，以为真让它学会了，嘴里还不停叫嚷着“博主塌房了”、“不信，这是AI”。. 这毫不拖泥带水的稍息动作一出，网友就了然了，原来，它依旧蒙题、依旧乱猜。好好好，既然如此那我问你，何棒之有？. 就这样，主人巧施连环计，开门误成大笨猫。网上随便溜达一圈，都有网友指着嘎嘎笑它智力不详。. 先不说大部分猫的大脑皮层如上图一样，光滑到几乎没有褶皱，单说开门这个个例。你可以说它算不上考研的料，但察言观色这一块，人情世故这一块，它完全是考公的好“猫”才。. 看这犹豫试探的小表情，没做对的时候还会撒娇两下，试图“萌”混过关。. 又或许是开门妈那一声声“蒸蚌”听着太洗脑，让人一看到视频就跟那个巴浦洛夫的狗听到铃声开始流口水一样，

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model="deepseek-chat",
    tools=[search_tool],
    system_prompt="你是一个智能助手，请使用工具来解决用户问题。"
)

In [31]:
from langchain.messages import HumanMessage
response = agent.invoke(
    {"messages":[HumanMessage(content="蒸蚌是什么梗?")]}
)

for message in response['messages']:
    print(message.pretty_print())


================================ Human Message =================================

蒸蚌是什么梗?
None
================================== Ai Message ==================================

让我帮你搜索一下这个网络梗的含义。
Tool Calls:
  tavily_search (call_00_53ztn9ClQsViSfzwJPEg7280)
 Call ID: call_00_53ztn9ClQsViSfzwJPEg7280
  Args:
    query: 蒸蚌 是什么梗 网络用语
None
================================= Tool Message =================================
Name: tavily_search

{"query": "蒸蚌 是什么梗 网络用语", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.facebook.com/61586315125667/posts/%E9%9A%A8%E4%BE%BF%E6%8C%87%E4%B8%80%E6%8C%87%E8%81%BD%E5%88%B0%E8%92%B8%E8%9A%8C%E4%BB%A3%E8%A1%A8%E6%8C%87%E5%B0%8D%E4%BA%86%E8%B2%93%E7%8B%97%E5%90%90%E6%A7%BD%E4%B8%BB%E4%BA%BA-%E8%B2%93%E7%8B%97%E9%9B%BB%E5%8F%B0-%E7%B1%B3%E8%80%81%E9%BC%A0%E7%9A%84%E6%84%8F%E6%80%9D-%E8%92%B8%E8%9A%8C-%E6%90%9E%E7%AC%91/122096350329210504/", "title": "隨便指一指，聽到蒸蚌代表指對了   #貓狗吐槽主人#貓狗電台#米老鼠 ...", "content": "這是在講一個最近很紅的網

In [32]:
from langchain_core.tools import tool
tavily=TavilySearch(
    max_results=5,
    topic="general"
)
@tool
def web_search(query:str):
    """Search the web for information."""
    return tavily.invoke(query)

In [33]:
from pydantic import BaseModel,Field
class Reference(BaseModel):
    title:str=Field(description="The title of the web page cited in the answer")
    url:str=Field(description="The url of the web page cited in the answer")

class AnswerInfo(BaseModel):
    answer:str=Field(description="The final answer for user")
    reference:list[Reference]=Field(description="The web page cited in the answer")

In [37]:
agent = create_agent(model="deepseek-chat", tools=[web_search], system_prompt="你是一个智能助手，请使用工具来解决用户问题。",
                 response_format=AnswerInfo)

In [38]:
response=agent.invoke(
    {"messages":HumanMessage(content="蒸蚌是什么梗?")}
)
print(response['structured_response'])

answer='**"蒸蚌"** 是一个网络流行梗，是 **"真棒"** 的谐音写法，意思是"真的很棒"。\n\n这个梗的起源主要来自一只名叫 **"大开门"** 的三花猫（网红宠物博主 @超级无敌大开门 的猫）。视频中，主人分别拿出萝卜和纸巾等物品让猫咪辨认，每当猫咪蒙对答案时，主人就会用高昂的声音喊出"真棒！"（听起来像"蒸蚌"），并奖励它冻干。\n\n由于猫咪其实根本不会分辨物品，全靠**察言观色**来蒙题（先试探地摸一个，看主人脸色不对就换一个），这种"智力不详但人情世故满分"的反差感极具喜感，加上主人魔性的"蒸蚌"声非常洗脑，迅速在网络上走红，引发了大量二创。\n\n后来"蒸蚌"逐渐演变为一个通用的网络用语，可以用来：\n1. **夸奖别人或自己**：表达"真棒"的意思，带有幽默和玩梗的色彩。\n2. **表达蒙对答案的喜悦**：比如考试蒙对题、瞎猜中奖等场景。\n\n相关话题在抖音等平台获得了超**18亿次**的播放量，成为现象级网络热梗。' reference=[Reference(title='我蒸蚌是什麼意思？ - HiNative', url='https://tw.hinative.com/questions/15904858'), Reference(title='一只笨猫学习辨认萝卜和纸巾 - 虎嗅', url='https://www.huxiu.com/article/4820519.html'), Reference(title='蒸蚌！全网最火小猫咪喜提全球代言 - SocialBeta', url='https://socialbeta.com/article/110971')]
